In [2]:
import pandas as pd 
import numpy as np 
import seaborn as sns 
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [3]:
df = pd.read_csv("../data.csv")

In [4]:
df_mod = df[:200000]
df_mod.shape

(200000, 11)

In [5]:
df_mod = pd.get_dummies(df_mod, columns=["type"])
# Borramos las columnas que no aportan información numérica
df_mod = df_mod.drop(['nameOrig', 'nameDest'], axis=1)

In [6]:
df_mod.sample(3)

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
65388,9,105378.98,20115.0,0.00,390985.07,984424.07,0,0,False,True,False,False,False
194944,13,229137.19,77174.0,306311.19,727372.55,827622.71,0,0,True,False,False,False,False
6399,6,155270.31,0.0,0.00,346277.88,618730.52,0,0,False,True,False,False,False


In [7]:
df_mod.info()

<class 'pandas.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 13 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   step            200000 non-null  int64  
 1   amount          200000 non-null  float64
 2   oldbalanceOrg   200000 non-null  float64
 3   newbalanceOrig  200000 non-null  float64
 4   oldbalanceDest  200000 non-null  float64
 5   newbalanceDest  200000 non-null  float64
 6   isFraud         200000 non-null  int64  
 7   isFlaggedFraud  200000 non-null  int64  
 8   type_CASH_IN    200000 non-null  bool   
 9   type_CASH_OUT   200000 non-null  bool   
 10  type_DEBIT      200000 non-null  bool   
 11  type_PAYMENT    200000 non-null  bool   
 12  type_TRANSFER   200000 non-null  bool   
dtypes: bool(5), float64(5), int64(3)
memory usage: 13.2 MB


In [8]:
X = df_mod.drop(['isFraud', 'isFlaggedFraud'], axis=1)

y = df_mod['isFraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

modelo_fraude = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')

modelo_fraude.fit(X_train, y_train)

print("Modelo entrenado")

Modelo entrenado


In [15]:
probabilidades = modelo_fraude.predict_proba(X_test)[:,1]
df_resultados = X_test.copy()
df_resultados['Fraude_Real'] = y_test
df_resultados['Probabilidad_Fraude'] = probabilidades
def mapear_escala_sospecha(prob):
    if prob < 0.30:
        return 'Muy Improbable'
    elif prob < 0.70:
        return 'Probable / Sospechosa'
    else:
        return 'Muy Probable'
df_resultados['Nivel_Alerta'] = df_resultados['Probabilidad_Fraude'].apply(mapear_escala_sospecha)
display(df_resultados[df_resultados["Nivel_Alerta"]=="Probable / Sospechosa"])

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER,Fraude_Real,Probabilidad_Fraude,Nivel_Alerta
5853,6,26768.50,26768.50,0.0,101976.00,128744.50,False,True,False,False,False,1,0.34,Probable / Sospechosa
131710,11,306030.46,257483.73,0.0,0.00,173109.47,False,False,False,False,True,0,0.31,Probable / Sospechosa
5489,5,97544.43,87120.00,0.0,0.00,97544.43,False,False,False,False,True,0,0.31,Probable / Sospechosa
6898,6,10565.00,10565.00,0.0,36275.00,24380.72,False,True,False,False,False,1,0.48,Probable / Sospechosa
4668,4,169941.73,169941.73,0.0,0.00,169941.73,False,True,False,False,False,1,0.53,Probable / Sospechosa
17664,8,92338.97,41121.00,0.0,0.00,0.00,False,False,False,False,True,0,0.31,Probable / Sospechosa
12394,7,408611.16,289437.80,0.0,918.00,0.00,False,False,False,False,True,0,0.37,Probable / Sospechosa
179625,12,21395.15,3014.00,0.0,0.00,21395.15,False,False,False,False,True,0,0.31,Probable / Sospechosa
34168,8,43092.00,43092.00,0.0,0.00,0.00,False,False,False,False,True,1,0.34,Probable / Sospechosa
20501,8,393.99,231.00,0.0,6253.00,6647.00,False,True,False,False,False,0,0.31,Probable / Sospechosa


In [ ]:
# # 1. Le pedimos al modelo las PROBABILIDADES en vez de la predicción fija (0 o 1)
# # [:, 1] nos da la probabilidad de que SÍ sea fraude (la clase 1)
# probabilidades = modelo_fraude.predict_proba(X_test)[:, 1]

# # 2. Creamos un nuevo DataFrame para ver los resultados finales codo a codo
# df_resultadoss = X_test.copy()
# df_resultadoss['Fraude_Real'] = y_test
# df_resultadoss['Probabilidad_Fraude'] = probabilidades

# # 3. Definimos tu función con los criterios de escala de riesgo
# def mapear_escala_sospecha(prob):
#     if prob < 0.30:
#         return 'Muy Improbable'
#     elif prob < 0.70:
#         return 'Probable / Sospechosa'
#     else:
#         return 'Muy Probable'

# # 4. Aplicamos la función para crear la columna con los niveles que querías
# df_resultadoss['Nivel_Alerta'] = df_resultadoss['Probabilidad_Fraude'].apply(mapear_escala_sospecha)

# # 5. Miramos una muestra de cómo quedó la clasificación
# print("--- MUESTRA DE TRANSACCIONES CLASIFICADAS ---")
# print(df_resultadoss[['amount', 'oldbalanceOrg', 'Probabilidad_Fraude', 'Nivel_Alerta', 'Fraude_Real']].head(10))

# print("\n--- DISTRIBUCIÓN DE ALERTAS GENERADAS ---")
# print(df_resultadoss['Nivel_Alerta'].value_counts())